# 🩺 MedLLM Fine-tune — QLoRA on Llama 3.2 3B (Unsloth Edition)
**Target: ~45–90 min on T4 GPU using Unsloth (2–5x faster than standard QLoRA)**

### Before running:
1. **Runtime → Change runtime type → T4 GPU**
2. Add Secrets: `HF_TOKEN` and `WANDB_API_KEY`
3. **Runtime → Run All**
---

## Step 1 — Install Unsloth + Dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets wandb
print('✅ Unsloth + dependencies installed')

## Step 2 — GPU Check & Authenticate

In [ ]:
import os, torch, wandb
from google.colab import userdata
from huggingface_hub import login

assert torch.cuda.is_available(), '❌ No GPU! Go to Runtime → Change runtime type → T4 GPU'
print(f'✅ GPU : {torch.cuda.get_device_name(0)}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Detect correct precision — T4 = fp16, A100/H100 = bf16
USE_BF16 = torch.cuda.is_bf16_supported()
print(f'   Precision: {"bf16" if USE_BF16 else "fp16 (T4)"}')

HF_TOKEN  = userdata.get('HF_TOKEN')
WANDB_KEY = userdata.get('WANDB_API_KEY')
if not HF_TOKEN:  raise ValueError('❌ HF_TOKEN missing in Secrets')
if not WANDB_KEY: raise ValueError('❌ WANDB_API_KEY missing in Secrets')

os.environ['HF_TOKEN']      = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_KEY
login(token=HF_TOKEN)
wandb.login(key=WANDB_KEY)
print('✅ Authenticated')

## Step 3 — Prepare Dataset

In [ ]:
import os
from datasets import load_dataset, Dataset, DatasetDict

def format_instruction(question, answer):
    return f"[INST] {question.strip()} [/INST] {answer.strip()}"

print('Loading lavita/MedQuAD...')
raw = load_dataset('lavita/MedQuAD')
df  = raw['train'].to_pandas()
print(f'Original rows: {len(df)}')

df['answer_len'] = df['answer'].astype(str).apply(lambda x: len(x.split()))
df = df[(df['answer_len'] >= 20) & (df['answer_len'] <= 200)].copy()
print(f'After filtering: {len(df)}')

df['text'] = df.apply(lambda r: format_instruction(r['question'], r['answer']), axis=1)
hf_dataset = Dataset.from_pandas(df[['text']], preserve_index=False)

split1 = hf_dataset.train_test_split(test_size=0.10, seed=42)
split2 = split1['test'].train_test_split(test_size=0.50, seed=42)
dataset = DatasetDict({
    'train':      split1['train'],
    'validation': split2['train'],
    'test':       split2['test'],
})
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

os.makedirs('eval', exist_ok=True)
df.sample(n=len(dataset['test']), random_state=42)[['question','answer']].to_json(
    'eval/ground_truth.jsonl', orient='records', lines=True)
print('✅ Dataset ready!')

## Step 4 — Load Model with Unsloth

In [ ]:
import os, torch
from unsloth import FastLanguageModel

HF_TOKEN = os.environ['HF_TOKEN']
MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
MAX_SEQ  = 512

print('Loading model with Unsloth (4-bit)...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ,
    dtype          = None,      # auto: fp16 on T4, bf16 on A100
    load_in_4bit   = True,
    token          = HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 8,
    lora_alpha                 = 16,
    target_modules             = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout               = 0,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',  # Unsloth optimized checkpointing
    random_state               = 42,
)
model.print_trainable_parameters()
print('✅ Model ready with Unsloth!')

## Step 5 — Fine-tune with SFTTrainer + Unsloth
> **Expected: ~1–2 sec/step → 800 steps ≈ 45–90 min on T4**

In [ ]:
import torch, wandb
from trl import SFTTrainer, SFTConfig

wandb.init(project='medllm-finetune', name='finetune-llama-3.2-3b-unsloth')

# Detect precision using torch — works regardless of Unsloth version
USE_BF16 = torch.cuda.is_bf16_supported()

sft_config = SFTConfig(
    output_dir                  = './results',
    save_strategy               = 'steps',
    save_steps                  = 400,
    save_total_limit            = 1,

    num_train_epochs            = 1,
    max_steps                   = 800,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,     # effective batch = 16
    gradient_checkpointing      = False, # Unsloth handles this internally
    optim                       = 'adamw_8bit',

    learning_rate               = 2e-4,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    max_grad_norm               = 0.3,

    eval_strategy               = 'no',

    bf16                        = USE_BF16,       # bf16 only on A100+
    fp16                        = not USE_BF16,   # fp16 on T4 (native tensor cores)

    logging_steps               = 25,
    report_to                   = 'wandb',

    dataloader_num_workers      = 0,
    dataloader_pin_memory       = False,

    max_length                  = 512,
    dataset_text_field          = 'text',
    packing                     = True,
)

trainer = SFTTrainer(
    model            = model,
    args             = sft_config,
    train_dataset    = dataset['train'],
    processing_class = tokenizer,
)

print(f'Training on {len(dataset["train"])} examples | max_steps=800 | Unsloth enabled')
print(f'Precision: {"bf16" if USE_BF16 else "fp16"} | expect 1-2 sec/step')
trainer.train()
print('✅ Training complete!')

## Step 6 — Save & Push to Hub

In [ ]:
import os, wandb

HF_TOKEN     = os.environ['HF_TOKEN']
ADAPTER_PATH = './medllm-lora'

trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'✅ Adapters saved to {ADAPTER_PATH}')

trainer.model.push_to_hub('Viresh2408/medllm-lora', token=HF_TOKEN)
tokenizer.push_to_hub('Viresh2408/medllm-lora', token=HF_TOKEN)
print('✅ Live at https://huggingface.co/Viresh2408/medllm-lora')

wandb.finish()
print('\n🎉 Fine-tuning Complete!')

## Step 7 — Test Fine-tuned Model

In [ ]:
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

HF_TOKEN     = os.environ['HF_TOKEN']
MODEL_ID     = 'meta-llama/Llama-3.2-3B-Instruct'
ADAPTER_PATH = './medllm-lora'
DTYPE        = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print('Loading fine-tuned model...')
base     = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map='auto', token=HF_TOKEN)
tok      = AutoTokenizer.from_pretrained(ADAPTER_PATH)
ft_model = PeftModel.from_pretrained(base, ADAPTER_PATH)
ft_model.eval()

question = 'What are the common symptoms of hypothyroidism?'
prompt   = f'[INST] {question} [/INST]'
inputs   = tok(prompt, return_tensors='pt').to(ft_model.device)

with torch.no_grad():
    output = ft_model.generate(
        **inputs, max_new_tokens=200, do_sample=True,
        temperature=0.1, top_p=0.9, pad_token_id=tok.eos_token_id,
    )

answer = tok.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f'\nQ: {question}\n\nMedLLM Answer:\n{answer}')